### CRISP-DM · Fase 6: Despliegue

Función de predicción para revisar registros nuevos de la estación, probada con registros de ejemplo.

- **Entradas:** `crisp_dm/models/svm.joblib`, `crisp_dm/data/interim/success_criteria.json`
- **Salida:** `crisp_dm/data/processed/sample_predictions.csv`: resultado con los registros de ejemplo

> **Caso de demostración:** los registros de ejemplo son inventados.

#### Librerías

In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd

#### Rutas

Relativas a la carpeta del notebook.

In [2]:
INTERIM_DIR = Path("../data/interim")
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")

#### Carga del modelo y del umbral de revisión

In [3]:
model = joblib.load(MODELS_DIR / "svm.joblib")
with open(INTERIM_DIR / "success_criteria.json", encoding="utf-8") as f:
    REVIEW_THRESHOLD = json.load(f)["review_threshold"]

NUMERIC = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
VALID_ISLANDS = ["Biscoe", "Dream", "Torgersen"]
VALID_SEX = ["MALE", "FEMALE"]

#### Función de revisión

Aplica la misma limpieza de la fase 3, rechaza islas desconocidas y devuelve:

- `especie_predicha` y `probabilidad`.
- `coincide`: si la predicción coincide con la especie anotada en campo.
- `revisar`: si el registro no coincide o la probabilidad es menor que el umbral.

In [4]:
def review_records(records):
    df = pd.DataFrame(records).copy()

    unknown_islands = set(df["island"]) - set(VALID_ISLANDS)
    if unknown_islands:
        raise ValueError(f"Islas fuera del alcance del modelo: {sorted(unknown_islands)}")
    if df[NUMERIC].isna().all(axis=1).any():
        raise ValueError("Hay registros sin ninguna medida")

    df["sex"] = df["sex"].where(df["sex"].isin(VALID_SEX), "UNKNOWN")

    proba = model.predict_proba(df[NUMERIC + ["island", "sex"]])
    df["especie_predicha"] = model.classes_[proba.argmax(axis=1)]
    df["probabilidad"] = proba.max(axis=1).round(3)
    df["coincide"] = df["especie_predicha"] == df["especie_anotada"]
    df["revisar"] = ~df["coincide"] | (df["probabilidad"] < REVIEW_THRESHOLD)
    return df

#### Registros de ejemplo

In [5]:
sample_records = [
    {"registro": "ejemplo_1", "especie_anotada": "Gentoo", "island": "Biscoe",
     "culmen_length_mm": 47.5, "culmen_depth_mm": 14.8, "flipper_length_mm": 217.0, "body_mass_g": 5100.0, "sex": "MALE"},
    {"registro": "ejemplo_2", "especie_anotada": "Adelie", "island": "Dream",
     "culmen_length_mm": 49.0, "culmen_depth_mm": 18.6, "flipper_length_mm": 196.0, "body_mass_g": 3750.0, "sex": "FEMALE"},
    {"registro": "ejemplo_3", "especie_anotada": "Adelie", "island": "Torgersen",
     "culmen_length_mm": 38.6, "culmen_depth_mm": 18.2, "flipper_length_mm": 190.0, "body_mass_g": 3600.0, "sex": None},
]

sample_predictions = review_records(sample_records)
sample_predictions[["registro", "especie_anotada", "especie_predicha", "probabilidad", "coincide", "revisar"]]

,registro,especie_anotada,especie_predicha,probabilidad,coincide,revisar
0,ejemplo_1,Gentoo,Gentoo,0.998,True,False
1,ejemplo_2,Adelie,Chinstrap,0.985,False,True
2,ejemplo_3,Adelie,Adelie,0.994,True,False


#### Rechazo de una isla fuera del alcance

In [6]:
try:
    review_records([{**sample_records[0], "island": "Anvers"}])
except ValueError as error:
    print(f"Rechazado: {error}")

Rechazado: Islas fuera del alcance del modelo: ['Anvers']


#### Guardado de resultados

In [7]:
sample_predictions.to_csv(PROCESSED_DIR / "sample_predictions.csv", index=False)
print(f"Guardado: {PROCESSED_DIR / 'sample_predictions.csv'}")

Guardado: ..\data\processed\sample_predictions.csv
